In [136]:
import pandas as pd
import re

In [137]:
data = pd.read_excel("Club2024.xlsx", sheet_name=None)

In [138]:
metadata = data.pop("Races", "Overview")

In [139]:
overview = data.pop("Overview")

In [140]:
NON_SHR = ["Blisco", "Perris Horseshoe"]
#Race data that doesn't fit the SHR table format (Usually the two british champs races

In [141]:
# Function to convert the original results to the desired format
def convert_blisco_format(df):
    # Convert 'Pos' from text like '5th', '11th' to integers like 5, 11
    df['Pos'] = df['Pos'].apply(lambda x: int(re.sub(r'\D', '', str(x))))  # Remove non-numeric characters
    
    # Function to round down age categories with a '5' in them (e.g., 'M45' -> 'M40')
    def adjust_category(cat):
        # Check if category is age-based (contains 'M' or 'F' and a number)
        if 'M' in cat or 'F' in cat:
            # Try to extract the numeric part and ensure it's valid
            num_part = cat.replace('M', '').replace('F', '')
            
            # If the numeric part is not empty and is a valid integer, proceed with rounding
            if num_part.isdigit():
                num = int(num_part)
                # Round down categories that have a '5' in them (e.g., M45 -> M40)
                if num % 10 == 5:
                    return cat[0] + str(num // 10 * 10)  # Round down to nearest multiple of 10
            return cat  # Return original category if no rounding needed
        return cat  # Return original category if not an age-based category


    # Map MSEN category to M and FSEN category to F
    category_map = {
        'MSEN': 'M',  # 'MSEN' becomes 'M'
        'FSEN': 'F',  # 'FSEN' becomes 'F'
    }

    # Apply category mapping for MSEN and FSEN
    df['Cat'] = df['Category'].map(category_map).fillna(df['Category'])

    # Adjust the 'Cat' for rounding down categories
    df['Cat'] = df['Cat'].apply(adjust_category)

    # Combine 'First name' and 'Surname' into 'Runner'
    df['Runner'] = df['Name']

    df['Club'] = df['Team']
    df['Club'] = df['Club'].fillna("Unattached")
    
    # Keep only necessary columns: 'Pos', 'Runner', 'Team', 'Cat', 'Time'
    df = df[['Pos', 'Runner', 'Club', 'Cat', 'Time']]
    
    # Ensure 'Time' remains in the same format (HH:MM:SS)
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.strftime('%H:%M:%S')

    return df

# Function to convert the data into the desired format
def convert_perris_format(df):
    # Remove rows where 'Position' contains '#'
    df = df[~df['Position'].str.contains('#', na=False)]
    
    # Combine 'First name' and 'Surname' into 'Runner'
    df['Runner'] = df['First name'] + ' ' + df['Surname']
    
    # Map Age Cat to appropriate categories
    def map_category(row):
        # If the Age Category is 'Open', just return the gender
        if row['Age Cat.'] == 'Open':
            return row['Gender']
        
        # Handle the 'U23' category (age under 40, map to M or F based on Gender)
        if row['Age Cat.'] == 'U23':
            return row['Gender']
        
        # Handle the cases with 5-year age categories (M45 -> M50, F45 -> F50, etc.)
        age_cat = row['Age Cat.']
        if re.match(r'\D*\d{2}$', age_cat):  # Matches something like M45, M50, F45, F50, etc.
            age = int(re.search(r'\d+', age_cat).group())  # Extract the number
            rounded_age = (age // 5) * 5  # Round down to the nearest multiple of 5
            category = f'{row["Gender"]}{rounded_age}'
            return category
        return row['Age Cat.']  # Return the original Age Cat. if it doesn't match the above pattern

    # Apply the map_category function to the dataframe
    df['Cat'] = df.apply(map_category, axis=1)
    df['Club'] = df['Club'].fillna("Unattached")

    # Keep only necessary columns: 'Position', 'Runner', 'Club', 'Cat', 'Time'
    df = df[['Position', 'Runner', 'Club', 'Cat', 'Time']]

    # Rename 'Position' to 'Pos', 'First name' to 'Runner' and 'Age Cat.' to 'Cat'
    df = df.rename(columns={'Position': 'Pos'})
    
    # Ensure 'Time' remains in the same format (HH:MM:SS)
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.strftime('%H:%M:%S')

    return df

# Function to add 'Cat Pos' for each dataframe
def add_cat_pos(df):
    # Convert Time column to timedelta
    #df['Time'] = pd.to_timedelta(df['Time'])
    # Rank by Category and Time
    df['Cat Pos'] = df.groupby('Cat')['Time'].rank(method='min', ascending=True).astype(int)

    # Return the dataframe with the new 'Cat Pos' column
    return df

# Loop through each dataframe in the dictionary and apply the 'Cat Pos' function
def get_carnethies(df):
    df = df[df["Club"].str.startswith("Carnethy")]
    return df
    
for race, results in data.items():
    if race in NON_SHR:
        if race == "Blisco":
            print(race, results.columns)
            results = convert_blisco_format(results)
        elif race == "Perris Horseshoe":
            results = convert_perris_format(results)
            print(race, results.columns)
            
    results = get_carnethies(results)
    data[race] = add_cat_pos(results)

Blisco Index(['Pos', 'No.', 'Name', 'Team', 'Category', 'Class', 'Time'], dtype='object')
Perris Horseshoe Index(['Pos', 'Runner', 'Club', 'Cat', 'Time'], dtype='object')


/tmp/ipykernel_4817/418097766.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cat Pos'] = df.groupby('Cat')['Time'].rank(method='min', ascending=True).astype(int)
/tmp/ipykernel_4817/418097766.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cat Pos'] = df.groupby('Cat')['Time'].rank(method='min', ascending=True).astype(int)
/tmp/ipykernel_4817/418097766.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

In [134]:
data["Blisco"]

,Pos,Runner,Club,Cat,Time,Cat Pos
0,5,Finn Lydon,Carnethy Hill Racing Club,M,00:40:02,1
1,11,Kieran Cooper,Carnethy Hill Racing Club,M,00:40:50,2
2,31,Csoban Balogh,Carnethy Hill Racing Club,M,00:42:26,3
3,32,Samuel Tosh,Carnethy Hill Racing Club,M,00:42:30,4
4,55,Paul Faulkner,Carnethy Hill Racing Club,M50,00:45:37,1
5,94,Michael Reid,Carnethy Hill Racing Club,M40,00:48:15,1
6,118,Stewart Whitlie,Carnethy Hill Racing Club,M60,00:51:20,1
7,153,Drew Sharkey,Carnethy Hill Racing Club,M50,00:54:51,2


In [135]:
# Function to create a table with the best 6 races and total points for each runner and category
def create_runner_points_table(data):
    # Initialize an empty dataframe to accumulate the results
    runner_points = pd.DataFrame(columns=['Runner', 'Cat', 'Cat Pos'])

    # Loop through each race in the data dictionary
    for race, df in data.items():
        #if race in NON_SHR:  # Skip certain races if needed
        #    continue

        # Ensure 'Cat Pos' is numeric (convert using astype(float))
        df['Cat Pos'] = df['Cat Pos'].astype(float)
        
        # Check if conversion worked and handle any invalid 'Cat Pos' values
        if df['Cat Pos'].isnull().any():
            print(f"Warning: Some 'Cat Pos' values are invalid in {race}, converting them to NaN")

        # Group by 'Runner' and 'Cat' and take 'Cat Pos' for each runner in each category
        race_points = df[['Runner', 'Cat', 'Cat Pos']]

        # Append the points to the accumulated table of runner points
        runner_points = pd.concat([runner_points, race_points], ignore_index=True)

    # For each runner and category, we want to get the best 6 results (lowest Cat Pos)
    runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
        lambda x: x.nsmallest(6, 'Cat Pos')  # Select the best 6 results (lowest Cat Pos)
    ).reset_index(drop=True)

    # Filter out runners who have fewer than 6 races
    runner_counts = runner_points_sorted.groupby(['Runner', 'Cat']).size()
    valid_runners = runner_counts[runner_counts >= 6].index

    # Filter the dataframe to only include valid runners (those with >= 6 races)
    runner_points_sorted = runner_points_sorted[
        runner_points_sorted.set_index(['Runner', 'Cat']).index.isin(valid_runners)
    ]

    # Now sum the Cat Pos for each runner and category (best 6 results)
    runner_points_sorted['Points'] = runner_points_sorted.groupby(['Runner', 'Cat'])['Cat Pos'].transform('sum')

    # Drop duplicates and only keep the first occurrence (because we already filtered for best 6 results)
    runner_points_sorted = runner_points_sorted.drop_duplicates(subset=['Runner', 'Cat'])

    # Sort by 'Cat' first, then by 'Points' (lowest is best)
    runner_points_sorted = runner_points_sorted.sort_values(by=['Cat', 'Points'], ascending=[True, True]).reset_index(drop=True)

    # Return only 'Runner', 'Cat', 'Cat Pos', and 'Points' columns for final output
    final_table = runner_points_sorted[['Runner', 'Cat', 'Cat Pos', 'Points']]

    return final_table

# Example usage (assuming 'data' is your dictionary of dataframes):
runner_points_table = create_runner_points_table(data)

# Display the table
pd.set_option('display.max_rows', None)
print(runner_points_table)


/tmp/ipykernel_4817/3695307936.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runner_points = pd.concat([runner_points, race_points], ignore_index=True)


            Runner  Cat  Cat Pos  Points
0      Aidan Smith    M      1.0    15.0
1     Iain Gilmore    M      1.0    29.0
2    Andrew Fallas  M40      1.0     6.0
3     Michael Reid  M40      1.0    11.0
4     Eliot Sedman  M40      1.0    12.0
5    Andrew Macrae  M50      1.0     6.0
6     Neil Gilmore  M60      1.0     6.0
7  Stewart Whitlie  M60      1.0     6.0


/tmp/ipykernel_4817/3695307936.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(


In [54]:
import pandas as pd

# Function to filter out runners who are from the Carnethy club
def get_carnethies(df):
    return df[df["Club"].str.startswith("Carnethy")]

# Function to create a table with the best 6 races and total points for each runner and category
def create_runner_points_table(data):
    # Initialize an empty dataframe to accumulate the results
    runner_points = pd.DataFrame(columns=['Runner', 'Cat', 'Cat Pos', 'Race'])

    # Loop through each race in the data dictionary
    for race, df in data.items():
        if race in NON_SHR:  # Skip certain races if needed
            continue

        # Ensure 'Cat Pos' is numeric (convert using astype(float))
        df['Cat Pos'] = df['Cat Pos'].astype(float)

        # Check if conversion worked and handle any invalid 'Cat Pos' values
        if df['Cat Pos'].isnull().any():
            print(f"Warning: Some 'Cat Pos' values are invalid in {race}, converting them to NaN")

        # Apply 'get_carnethies' to filter only Carnethy club runners
        df = get_carnethies(df)

        # Add race column to the dataframe
        race_points = df[['Runner', 'Cat', 'Cat Pos']].copy()
        race_points['Race'] = race  # Add the race name to each entry

        # Append the points to the accumulated table of runner points
        runner_points = pd.concat([runner_points, race_points], ignore_index=True)

    # Now, we need to calculate the 'Ran' column, which counts how many races a runner participated in.
    runner_counts = runner_points.groupby(['Runner', 'Cat'])['Cat Pos'].count()

    # Filter out runners with fewer than 6 races
    valid_runners = runner_counts[runner_counts >= 6].index

    # Filter the runner_points dataframe to only include valid runners (those with >= 6 races)
    runner_points = runner_points[runner_points.set_index(['Runner', 'Cat']).index.isin(valid_runners)]

    # For each runner and category, we want to get the best 6 results (lowest Cat Pos)
    runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
        lambda x: x.nsmallest(6, 'Cat Pos')  # Select the best 6 results (lowest Cat Pos)
    ).reset_index(drop=True)

    # Pivot the data so that each race becomes a column, and each runner gets their position in that race
    runner_points_pivot = runner_points_sorted.pivot_table(
        index=['Runner', 'Cat'], 
        columns='Race', 
        values='Cat Pos', 
        aggfunc='first'
    ).reset_index()

    # Calculate the 'Ran' column as the number of non-null Cat Pos values
    runner_points_pivot['Ran'] = runner_points_pivot.notnull().sum(axis=1) - 2  # Subtract 2 for 'Runner' and 'Cat' columns

    # Calculate the 'Total' as the sum of the best 6 race positions for each runner (sum of Cat Pos values)
    runner_points_pivot['Total'] = runner_points_pivot.drop(columns=['Runner', 'Cat', 'Ran']).sum(axis=1)

    # Rank runners by 'Total' points (lowest sum of Cat Pos is better)
    runner_points_pivot['Posn'] = runner_points_pivot['Total'].rank(method='min', ascending=True)

    # Fill NaN values (for races where the runner didn't participate) with 0 or empty strings
    runner_points_pivot = runner_points_pivot.fillna(0)

    # Select the final columns for output
    final_table = runner_points_pivot[['Posn', 'Runner', 'Ran'] + [col for col in runner_points_pivot.columns if col not in ['Posn', 'Runner', 'Ran', 'Total']]+['Total']]

    return final_table

# Example usage (assuming 'data' is your dictionary of dataframes):
runner_points_table = create_runner_points_table(data)

# Display the table
pd.set_option('display.max_rows', None)
runner_points_table


/tmp/ipykernel_4817/3895230727.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runner_points = pd.concat([runner_points, race_points], ignore_index=True)
/tmp/ipykernel_4817/3895230727.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(


Race,Posn,Runner,Ran,Cat,Allermuir,Arrochar,Caerketton,Caerketton Down,Cobbler,Mamores,Manor Water,Moffat,Run O Mill,Skyloop,Tap O North,Tinto,Traprain,Total
0,5.0,Aidan Smith,6,M,3.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,3.0,1.0,15.0
1,1.0,Andrew Macrae,6,M50,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,6.0
2,4.0,Eliot Sedman,6,M40,0.0,0.0,0.0,0.0,3.0,0.0,0.0,2.0,2.0,2.0,2.0,1.0,0.0,12.0
3,6.0,Iain Gilmore,6,M,0.0,8.0,0.0,0.0,8.0,5.0,1.0,0.0,2.0,0.0,0.0,5.0,0.0,29.0
4,1.0,Neil Gilmore,6,M60,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,6.0
5,3.0,Stewart Whitlie,6,M60,1.0,1.0,0.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,7.0


In [57]:
def create_runner_points_table(data):
    # Initialize an empty dataframe to accumulate the results
    runner_points = pd.DataFrame(columns=['Runner', 'Cat', 'Cat Pos', 'Race'])

    # Counter to track how many races each runner has participated in
    runner_race_counts = {}

    # Loop through each race in the data dictionary
    for race, df in data.items():
        if race in NON_SHR:  # Skip certain races if needed
            continue

        # Ensure 'Cat Pos' is numeric (convert using astype(float))
        df['Cat Pos'] = df['Cat Pos'].astype(float)

        # Check if conversion worked and handle any invalid 'Cat Pos' values
        if df['Cat Pos'].isnull().any():
            print(f"Warning: Some 'Cat Pos' values are invalid in {race}, converting them to NaN")

        # Apply 'get_carnethies' to filter only Carnethy club runners
        df = get_carnethies(df)

        # Add race column to the dataframe
        race_points = df[['Runner', 'Cat', 'Cat Pos']].copy()
        race_points['Race'] = race  # Add the race name to each entry

        # Append the points to the accumulated table of runner points
        runner_points = pd.concat([runner_points, race_points], ignore_index=True)

        # Update the race count for each runner
        for runner in df['Runner']:
            if runner not in runner_race_counts:
                runner_race_counts[runner] = 0
            runner_race_counts[runner] += 1

    # Now, we need to filter out runners who participated in fewer than 6 races
    eligible_runners = [runner for runner, count in runner_race_counts.items() if count >= 6]

    # Filter the runner_points dataframe to only include valid runners (those with >= 6 races)
    runner_points = runner_points[runner_points['Runner'].isin(eligible_runners)]

    # For each runner and category, we want to get the best 6 results (lowest Cat Pos)
    runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
        lambda x: x.nsmallest(6, 'Cat Pos')  # Select the best 6 results (lowest Cat Pos)
    ).reset_index(drop=True)

    # Pivot the data so that each race becomes a column, and each runner gets their position in that race
    runner_points_pivot = runner_points_sorted.pivot_table(
        index=['Runner', 'Cat'], 
        columns='Race', 
        values='Cat Pos', 
        aggfunc='first'
    ).reset_index()

    # Calculate the 'Ran' column as the number of non-null Cat Pos values
    runner_points_pivot['Ran'] = runner_points_pivot.notnull().sum(axis=1) - 2  # Subtract 2 for 'Runner' and 'Cat' columns

    # Calculate the 'Total' as the sum of the best 6 race positions for each runner (sum of Cat Pos values)
    runner_points_pivot['Total'] = runner_points_pivot.drop(columns=['Runner', 'Cat', 'Ran']).sum(axis=1)

    # Rank runners by 'Total' points (lowest sum of Cat Pos is better)
    runner_points_pivot['Posn'] = runner_points_pivot['Total'].rank(method='min', ascending=True)

    # Fill NaN values (for races where the runner didn't participate) with 0 or empty strings
    runner_points_pivot = runner_points_pivot.fillna(0)

    # Select the final columns for output
    final_table = runner_points_pivot[['Posn', 'Runner', 'Ran'] + [col for col in runner_points_pivot.columns if col not in ['Posn', 'Runner', 'Ran', 'Total']]+['Total']]

    return final_table

# Example usage (assuming 'data' is your dictionary of dataframes):
runner_points_table = create_runner_points_table(data)

# Display the table
pd.set_option('display.max_rows', None)
runner_points_table

/tmp/ipykernel_4817/2218039587.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runner_points = pd.concat([runner_points, race_points], ignore_index=True)
/tmp/ipykernel_4817/2218039587.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(


Race,Posn,Runner,Ran,Cat,Allermuir,Arrochar,Caerketton,Caerketton Down,Cobbler,Mamores,Manor Water,Moffat,Run O Mill,Skyloop,Tap O North,Tinto,Traprain,Total
0,5.0,Aidan Smith,6,M,3.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,3.0,1.0,15.0
1,1.0,Andrew Macrae,6,M50,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,6.0
2,4.0,Eliot Sedman,6,M40,0.0,0.0,0.0,0.0,3.0,0.0,0.0,2.0,2.0,2.0,2.0,1.0,0.0,12.0
3,6.0,Iain Gilmore,6,M,0.0,8.0,0.0,0.0,8.0,5.0,1.0,0.0,2.0,0.0,0.0,5.0,0.0,29.0
4,1.0,Neil Gilmore,6,M60,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,6.0
5,3.0,Stewart Whitlie,6,M60,1.0,1.0,0.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,7.0


In [142]:
# Function to create a table with the best 6 races and total points for each runner and category
def create_runner_points_table(data):
    # Initialize an empty dataframe to accumulate the results
    runner_points = pd.DataFrame(columns=['Runner', 'Cat', 'Cat Pos'])

    # Dictionary to store the total number of races for each runner
    runner_race_count = {}

    # Loop through each race in the data dictionary
    for race, df in data.items():
        # Ensure 'Cat Pos' is numeric (convert using astype(float))
        df['Cat Pos'] = df['Cat Pos'].astype(float)

        # Check if conversion worked and handle any invalid 'Cat Pos' values
        if df['Cat Pos'].isnull().any():
            print(f"Warning: Some 'Cat Pos' values are invalid in {race}, converting them to NaN")

        # Group by 'Runner' and 'Cat' and take 'Cat Pos' for each runner in each category
        race_points = df[['Runner', 'Cat', 'Cat Pos']]

        # Append the points to the accumulated table of runner points
        runner_points = pd.concat([runner_points, race_points], ignore_index=True)

        # Update the race count for each runner
        for runner in df['Runner']:
            if runner not in runner_race_count:
                runner_race_count[runner] = 0
            runner_race_count[runner] += 1

    # For each runner and category, we want to get the best 6 results (lowest Cat Pos)
    runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
        lambda x: x.nsmallest(6, 'Cat Pos')  # Select the best 6 results (lowest Cat Pos)
    ).reset_index(drop=True)

    # Filter out runners who have fewer than 6 races
    runner_counts = runner_points_sorted.groupby(['Runner', 'Cat']).size()
    valid_runners = runner_counts[runner_counts >= 6].index

    # Filter the dataframe to only include valid runners (those with >= 6 races)
    runner_points_sorted = runner_points_sorted[
        runner_points_sorted.set_index(['Runner', 'Cat']).index.isin(valid_runners)
    ]

    # Now sum the Cat Pos for each runner and category (best 6 results)
    runner_points_sorted['Points'] = runner_points_sorted.groupby(['Runner', 'Cat'])['Cat Pos'].transform('sum')

    # Drop duplicates and only keep the first occurrence (because we already filtered for best 6 results)
    runner_points_sorted = runner_points_sorted.drop_duplicates(subset=['Runner', 'Cat'])

    # Sort by 'Cat' first, then by 'Points' (lowest is best)
    runner_points_sorted = runner_points_sorted.sort_values(by=['Cat', 'Points'], ascending=[True, True]).reset_index(drop=True)

    # Add the total number of races each runner has done
    runner_points_sorted['Total Races'] = runner_points_sorted['Runner'].apply(lambda x: runner_race_count.get(x, 0))

    # Return only 'Runner', 'Cat', 'Cat Pos', 'Points', and 'Total Races' columns for final output
    final_table = runner_points_sorted[['Runner', 'Cat', 'Cat Pos', 'Points', 'Total Races']]

    return final_table

# Example usage (assuming 'data' is your dictionary of dataframes):
runner_points_table = create_runner_points_table(data)

# Display the table
pd.set_option('display.max_rows', None)
print(runner_points_table)

/tmp/ipykernel_4817/2615248119.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runner_points = pd.concat([runner_points, race_points], ignore_index=True)


            Runner  Cat  Cat Pos  Points  Total Races
0      Aidan Smith    M      1.0    15.0            7
1     Iain Gilmore    M      1.0    29.0            7
2    Andrew Fallas  M40      1.0     6.0            6
3     Michael Reid  M40      1.0    11.0            7
4     Eliot Sedman  M40      1.0    12.0            7
5    Andrew Macrae  M50      1.0     6.0            7
6     Neil Gilmore  M60      1.0     6.0            7
7  Stewart Whitlie  M60      1.0     6.0            8


/tmp/ipykernel_4817/2615248119.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
